# Verify `create_Hubbard` and `TamFermion.jl`'s `HubbardMomentumBasis` against XDiag ED

Takes the XDiag-produced data in
`/home/jek354/research/UCC_Hubbard/XDiag ED pipeline/ED_data2`
(the momentum-Slater ground states written by `consolidateConvertHubbardED_XDiag.py`) and
checks it two independent ways, with no basis conversion needed for either:

1. **`ed_functions.jl`'s `create_Hubbard`** builds the full real-space Hamiltonian; its
   lowest eigenvalue is compared directly to XDiag's global ground energy. No state vector
   needed.
2. **`TamFermion.jl`'s `HubbardMomentumBasis`** builds the Hamiltonian directly in the
   momentum-Slater basis. Its basis states are packed bitstrings of occupied momentum
   modes -- the exact same encoding the `.h5` file's `metadata/basis_labels` already use
   (both follow the row-major/C-order convention, per `ravel_c`/`unravel_c` in
   `TamLib.jl`) -- so the eigenvector stored in `data/evecs` can be read and placed
   directly into this basis with a plain integer lookup, and `<psi|H|psi>` compared
   straight to `data/energies`.

All six momentum sectors match XDiag to machine precision (previously, sectors where
`2*q_a/L_a` was not an integer for some axis showed a large mismatch -- this was traced to
a momentum-sign bug in XDiag's own `momentum_representation` (in
`.../UCC_Hubbard/XDiag ED pipeline/HubbardLib_XDiag.jl`), fixed 2026-09-01, with
`ED_data2` regenerated afterward; see that file's docstring for the full derivation and
the decisive test that found it).


In [ ]:
using LinearAlgebra
using SparseArrays
using HDF5
using Lattices
using Printf

const ED_DIR = "/home/jek354/research/ML-signproblem/experimenting/ed"
# const ED_DATA2_DIR = "/home/jek354/research/UCC_Hubbard/Python ED pipeline/ED_data/HubbardED_python_momSlater_3x2_nu_2_nd_2_t_1_m_2.h5"

# momslater_path = only(filter(f -> startswith(f, "HubbardED_XDiag_momSlater") && endswith(f, ".h5"),
#                               readdir(ED_DATA2_DIR)))]
momslater_path = "/home/jek354/research/UCC_Hubbard/Python ED pipeline/ED_data/HubbardED_python_momSlater_3x2_nu_3_nd_2_t_1_m_2.h5"
momslater_path = joinpath(ED_DATA2_DIR, momslater_path)
println("Reading: ", momslater_path)

Reading: /home/jek354/research/UCC_Hubbard/Python ED pipeline/ED_data/HubbardED_python_momSlater_3x2_nu_3_nd_2_t_1_m_2.h5


In [27]:
# TamFermion.jl's HubbardMomentumBasis
module WrapperMod
    ED_DIR = "/home/jek354/research/ML-signproblem/experimenting/ed"
    include(joinpath(ED_DIR, "TamLib.jl"))
    include(joinpath(ED_DIR, "TamFermion.jl"))
    using .TamLib
    using .TamFermion
end
using .WrapperMod.TamFermion: HubbardMomentumBasis

# ed_functions.jl's create_Hubbard
include(joinpath(ED_DIR, "utility_functions.jl"))
using .UtilityFunctions
using .UtilityFunctions: @safe_threads
include(joinpath(ED_DIR, "ed_objects.jl"))
include(joinpath(ED_DIR, "ed_functions.jl"))

build_save_name_prefix

In [32]:
# NOTE: HDF5.jl reads a dataset's dimensions in *reverse* of the shape h5py/numpy
# report (row-major on disk -> column-major in Julia), e.g. a (numU, m_s, dim_s)
# python array comes back as (dim_s, m_s, numU) here.
Lvec, nu, nd, uvec, qi_list = h5open(momslater_path, "r") do f
    Lvec = Int.(read(f["metadata/Lvec"]))
    nu = Int(read(f["metadata/nu"]))
    nd = Int(read(f["metadata/nd"]))
    uvec = read(f["data/uvec"])
    qi_list = sort(parse.(Int, keys(f["data/energies"])))
    (Lvec, nu, nd, uvec, qi_list)
end
t = 1.0  # ED_data2 was generated at t = 1
println("Lvec=$Lvec nu=$nu nd=$nd t=$t")
println("u values: ", uvec)
println("sectors (qi) present: ", qi_list)

Lvec=[3, 2] nu=3 nd=2 t=1.0
u values: [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5, 2.75, 3.0, 3.25, 3.5, 3.75, 4.0, 4.25, 4.5, 4.75, 5.0, 5.25, 5.5, 5.75, 6.0, 6.25, 6.5, 6.75, 7.0, 7.25, 7.5, 7.75, 8.0, 8.25, 8.5, 8.75, 9.0, 9.25, 9.5, 9.75, 10.0, 10.25, 10.5, 10.75, 11.0, 11.25, 11.5, 11.75, 12.0, 12.25, 12.5, 12.75, 13.0, 13.25, 13.5, 13.75, 14.0, 14.25, 14.5, 14.75, 15.0]
sectors (qi) present: [0, 1, 2, 3, 4, 5]


In [33]:
# --- Check 1: ed_functions.jl's create_Hubbard (full real-space Hamiltonian) ---
# No state vector needed: just diagonalize and compare the lowest eigenvalue to
# XDiag's global ground energy (the min energy over all sectors, for each u).
lattice = Square(Tuple(Lvec), Periodic())
subspace = HubbardSubspace(nu, nd, lattice)

for u in uvec
    Hm = HubbardModel(t, u, 0.0, false)
    H_ed = create_Hubbard(Hm, subspace)
    E_ed = minimum(eigvals(Matrix(Hermitian(H_ed))))

    E0_global = h5open(momslater_path, "r") do f
        u_idx = argmin(abs.(uvec .- u))
        minimum(read(f["data/energies/$qi"])[1, u_idx] for qi in qi_list)
    end

    @printf("u=%.3f  E_ed=%.10f  E0(XDiag, global gs)=%.10f  diff=%.2e\n",
            u, E_ed, E0_global, E_ed - E0_global)
    @assert abs(E_ed - E0_global) < 1e-8
end

u=0.250  E_ed=-10.7544547610  E0(XDiag, global gs)=-10.7544547610  diff=-2.49e-14
u=0.500  E_ed=-10.5178047288  E0(XDiag, global gs)=-10.5178047288  diff=-1.07e-14
u=0.750  E_ed=-10.2899911383  E0(XDiag, global gs)=-10.2899911383  diff=-8.88e-15
u=1.000  E_ed=-10.0709073741  E0(XDiag, global gs)=-10.0709073741  diff=-5.51e-14
u=1.250  E_ed=-9.8604026661  E0(XDiag, global gs)=-9.8604026661  diff=3.55e-15
u=1.500  E_ed=-9.6582871195  E0(XDiag, global gs)=-9.6582871195  diff=0.00e+00
u=1.750  E_ed=-9.4643376601  E0(XDiag, global gs)=-9.4643376601  diff=8.88e-15
u=2.000  E_ed=-9.2783044541  E0(XDiag, global gs)=-9.2783044541  diff=1.42e-14
u=2.250  E_ed=-9.0999173853  E0(XDiag, global gs)=-9.0999173853  diff=-1.78e-14
u=2.500  E_ed=-8.9288922420  E0(XDiag, global gs)=-8.9288922420  diff=-6.93e-14
u=2.750  E_ed=-8.7649363563  E0(XDiag, global gs)=-8.7649363563  diff=-4.09e-14
u=3.000  E_ed=-8.6077535346  E0(XDiag, global gs)=-8.6077535346  diff=-1.78e-15
u=3.250  E_ed=-8.4570482022  E0(XDia

In [34]:
# --- Check 2: TamFermion.jl's HubbardMomentumBasis, state read directly from the file ---
# basis_sector tells HubbardMomentumBasis to order its Hamiltonian to match the file's
# own basis_labels order, so the eigenvector V can be used directly -- no lookup dict.
N_sites = prod(Lvec)

@printf("%6s %4s %16s %16s %12s\n", "u", "qi", "E0 (XDiag)", "E_mom", "diff")
results = NamedTuple[]
for u in uvec
    u_idx = argmin(abs.(uvec .- u))
    for qi in qi_list
        E0, labels, V = h5open(momslater_path, "r") do f
            E0 = read(f["data/energies/$qi"])[1, u_idx]           # ground energy in this sector
            labels = read(f["metadata/basis_labels/$qi"])         # (2, dim) -> row 1 = up, row 2 = dn
            V = read(f["data/evecs/$qi"])[:, 1, u_idx]             # (dim,) ground eigenvector
            (E0, labels, V)
        end
        basis_sector = [UInt(labels[1, row]) | (UInt(labels[2, row]) << N_sites) for row in axes(labels, 2)]

        H_mom = HubbardMomentumBasis(t, u, Lvec, (nu, nd);
                                      q_target=qi, basis_sector=basis_sector, returnBasis=false)

        psi = V ./ norm(V)
        E_mom = real(dot(psi, H_mom * psi))

        @printf("%6.3f %4d %16.10f %16.10f %12.2e\n", u, qi, E0, E_mom, E_mom - E0)
        push!(results, (u=u, qi=qi, E0=E0, E_mom=E_mom, diff=E_mom - E0))
    end
end

tol = 1e-8
@assert all(abs(r.diff) < tol for r in results)
println("\nAll (u, sector) pairs match XDiag's energy to within $tol.")

     u   qi       E0 (XDiag)            E_mom         diff
 0.250    0    -7.7556004170    -7.7556004170     0.00e+00
 0.250    1    -9.8349231155    -9.8349231155     7.11e-15
 0.250    2   -10.7544547610   -10.7544547610    -5.33e-15
 0.250    3    -9.7544490176    -9.7544490176    -8.88e-15
 0.250    4   -10.7544547610   -10.7544547610    -3.55e-15
 0.250    5    -9.7544490176    -9.7544490176     0.00e+00
 0.500    0    -7.5223291187    -7.5223291187     4.44e-15
 0.500    1    -9.6730777326    -9.6730777326     0.00e+00
 0.500    2   -10.5178047288   -10.5178047288     1.78e-15
 0.500    3    -9.5177629303    -9.5177629303     1.78e-15
 0.500    4   -10.5178047288   -10.5178047288     3.55e-15
 0.500    5    -9.5177629303    -9.5177629303     3.55e-15
 0.750    0    -7.3000103905    -7.3000103905    -6.22e-15
 0.750    1    -9.5145378726    -9.5145378726    -3.55e-15
 0.750    2   -10.2899911383   -10.2899911383     3.55e-15
 0.750    3    -9.2898757105    -9.2898757105     0.00e+